# 长期记忆-基础API的使用

## 1、put()/get()的使用

### 1.1 基于InMemoryStore

In [3]:
from langgraph.store.memory import InMemoryStore

store = InMemoryStore()

namespace = ("user",)
user_id = "user_1"
user_name = "小明"

store.put(namespace, user_id, {"name":user_name})

item = store.get(namespace, user_id)
print(item)



Item(namespace=['user'], key='user_1', value={'name': '小明'}, created_at='2026-07-25T14:00:12.577228+00:00', updated_at='2026-07-25T14:00:12.577230+00:00')


更新数据：

In [5]:
user_name = "小红"

store.put(namespace, user_id, {"name":user_name})

item = store.get(namespace, user_id)
print(item)

Item(namespace=['user'], key='user_1', value={'name': '小红'}, created_at='2026-07-25T14:06:02.013247+00:00', updated_at='2026-07-25T14:06:02.013249+00:00')


## 2、基于PostgresStore

In [ ]:
from langgraph.store.postgres import PostgresStore

DB_URL = "you_DB_URL"
with PostgresStore.from_conn_string(DB_URL) as store:
    store.setup()

    namespace = ("user",)
    user_id = "user_1"
    user_name = "小明"

    store.put(namespace, user_id, {"name":user_name})

    item = store.get(namespace, user_id)
    print(item)

更新：

In [ ]:
with PostgresStore.from_conn_string(DB_URL) as store:
    store.setup()

    user_name = "小骚杠"

    store.put(namespace, user_id, {"name":user_name})

    item = store.get(namespace, user_id)
    print(item)

## 2、search()的使用

基于内存中数据的存储，进行演示

### 2.1 按照namespace前缀搜索

In [31]:
from langgraph.store.memory import InMemoryStore

# 初始化内存KV存储容器
store = InMemoryStore()

# Alice 用户偏好数据
namespace1 = ("users", "Alice", "memories")
key1 = 'preferences'
value1 = {
    "course": "计算机组成原理",
    "sports": "跑步",
    "food": "紫光园奶皮子酸奶"
}

# Bob 用户偏好数据
namespace2 = ("users", "Bob", "memories")
key2 = 'preferences'
value2 = {
    "course": "数字电路与模拟电路",
    "sports": "跑步",
    "food": "奶皮子糖葫芦"
}

# Black 用户偏好数据
namespace3 = ("users", "Black", "memories")
key3 = 'preferences'
value3 = {
    "course": "数字电路与模拟电路",
    "sports": "羽毛球",
    "food": "紫光园奶皮子酸奶"
}

# 将三组用户数据写入langgraph内存存储
store.put(namespace1, key1, value1)
store.put(namespace2, key2, value2)
store.put(namespace3, key3, value3)

In [32]:
for item in store.search(("users",)):
    print(item)

Item(namespace=['users', 'Alice', 'memories'], key='preferences', value={'course': '计算机组成原理', 'sports': '跑步', 'food': '紫光园奶皮子酸奶'}, created_at='2026-07-25T14:32:15.191677+00:00', updated_at='2026-07-25T14:32:15.191679+00:00', score=None)
Item(namespace=['users', 'Bob', 'memories'], key='preferences', value={'course': '数字电路与模拟电路', 'sports': '跑步', 'food': '奶皮子糖葫芦'}, created_at='2026-07-25T14:32:15.191708+00:00', updated_at='2026-07-25T14:32:15.191708+00:00', score=None)
Item(namespace=['users', 'Black', 'memories'], key='preferences', value={'course': '数字电路与模拟电路', 'sports': '羽毛球', 'food': '紫光园奶皮子酸奶'}, created_at='2026-07-25T14:32:15.191735+00:00', updated_at='2026-07-25T14:32:15.191735+00:00', score=None)


In [33]:
for item in store.search(("users","Bob")):
    print(item)

Item(namespace=['users', 'Bob', 'memories'], key='preferences', value={'course': '数字电路与模拟电路', 'sports': '跑步', 'food': '奶皮子糖葫芦'}, created_at='2026-07-25T14:32:15.191708+00:00', updated_at='2026-07-25T14:32:15.191708+00:00', score=None)


### 2.2 按照filter过滤

In [35]:

for item in store.search(("users",), filter={"sports": "跑步"}):
    print(item)

Item(namespace=['users', 'Alice', 'memories'], key='preferences', value={'course': '计算机组成原理', 'sports': '跑步', 'food': '紫光园奶皮子酸奶'}, created_at='2026-07-25T14:32:15.191677+00:00', updated_at='2026-07-25T14:32:15.191679+00:00', score=None)
Item(namespace=['users', 'Bob', 'memories'], key='preferences', value={'course': '数字电路与模拟电路', 'sports': '跑步', 'food': '奶皮子糖葫芦'}, created_at='2026-07-25T14:32:15.191708+00:00', updated_at='2026-07-25T14:32:15.191708+00:00', score=None)


In [36]:

for item in store.search(("users",), filter={"sports": "跑步1"}):
    print(item)

## 2.3 按照语义搜索

举例1：自定义嵌入函数（将一段文本转换为一个多维向量的函数）

In [38]:
from langgraph.store.memory import InMemoryStore

# 自定义嵌入函数：接收文本列表，返回对应向量列表，向量维度固定6维
def embed(text: list[str]) -> list[list[float]]:
    return [[1.0] * 6 for _ in range(len(text))]

# 向量索引配置
index_config = {
    "embed": embed,       # 文本向量化函数
    "dims": 6,            # 输出向量维度大小
    "fields": ["$", "course"]  # 需要构建向量索引的字段
}

# 初始化带向量检索能力的内存存储
store = InMemoryStore(
    index = index_config
)

# Alice 用户偏好记忆
namespace1 = ("users", "Alice", "memories")
key1 = 'preferences'
value1 = {
    "course": "计算机组成原理",
    "sports": "跑步",
    "food": "紫光园奶皮子酸奶"
}

# Bob 用户偏好记忆
namespace2 = ("users", "Bob", "memories")
key2 = 'preferences'
value2 = {
    "course": "数字电路与模拟电路",
    "sports": "跑步",
    "food": "奶皮子糖葫芦"
}

# Black 用户偏好记忆
namespace3 = ("users", "Black", "memories")
key3 = 'preferences'
value3 = {
    "course": "数字电路与模拟电路",
    "sports": "羽毛球",
    "food": "紫光园奶皮子酸奶"
}

# 写入三条用户记忆数据，写入时自动按照index配置生成向量索引
store.put(namespace1, key1, value1)
store.put(namespace2, key2, value2)
store.put(namespace3, key3, value3)

查看嵌入向量

In [39]:
from pprint import pprint
pprint(store._vectors)

defaultdict(<function InMemoryStore.__init__.<locals>.<lambda> at 0x000001DC15934EA0>,
            {('users', 'Alice', 'memories'): defaultdict(<class 'dict'>,
                                                         {'preferences': {'$': [1.0,
                                                                                1.0,
                                                                                1.0,
                                                                                1.0,
                                                                                1.0,
                                                                                1.0],
                                                                          'course': [1.0,
                                                                                     1.0,
                                                                                     1.0,
                                                           

In [40]:
from pprint import pprint
pprint(store._vectors[('users', 'Alice', 'memories')]['preferences']['$'])

[1.0, 1.0, 1.0, 1.0, 1.0, 1.0]


举例2：使用嵌入模型

In [41]:
from langgraph.store.memory import InMemoryStore
from dotenv import load_dotenv
from langchain.embeddings import init_embeddings
import os

# 加载.env环境变量文件
load_dotenv(override=True)

# 初始化OpenAI官方大维度嵌入模型
embedding_model = init_embeddings(
    model="openai:embedding-3",
    api_key=os.getenv("ZHIPUAI_API_KEY"),
    base_url=os.getenv("ZHIPUAI_BASE_URL"),
)

# 向量索引配置
index_config = {
    "embed": embedding_model,   # 真实OpenAI嵌入模型做文本向量化
    "dims": 3072,              # text-embedding-3-large 固定输出向量维度3072
    "fields": ["$"]            # 将整条用户偏好字典整体序列化后生成向量
}

# 开启向量检索的内存存储
store = InMemoryStore(index=index_config)

# Alice 用户个人偏好记忆
namespace1 = ("users", "Alice", "memories")
key1 = 'preferences'
value1 = {
    "course": "计算机组成原理",
    "sports": "跑步",
    "food": "紫光园奶皮子酸奶"
}

# Bob 用户个人偏好记忆
namespace2 = ("users", "Bob", "memories")
key2 = 'preferences'
value2 = {
    "course": "数字电路与模拟电路",
    "sports": "跑步",
    "food": "奶皮子糖葫芦"
}

# Black 用户个人偏好记忆
namespace3 = ("users", "Black", "memories")
key3 = 'preferences'
value3 = {
    "course": "数字电路与模拟电路",
    "sports": "羽毛球",
    "food": "紫光园奶皮子酸奶"
}

# 写入记忆，自动生成语义向量存入索引，支持后续相似度检索
store.put(namespace1, key1, value1)
store.put(namespace2, key2, value2)
store.put(namespace3, key3, value3)

In [43]:
for item in store.search(("users", ), query="紫光园"):
        print(item)

Item(namespace=['users', 'Alice', 'memories'], key='preferences', value={'course': '计算机组成原理', 'sports': '跑步', 'food': '紫光园奶皮子酸奶'}, created_at='2026-07-25T14:49:18.172824+00:00', updated_at='2026-07-25T14:49:18.172828+00:00', score=0.6291862088517552)
Item(namespace=['users', 'Black', 'memories'], key='preferences', value={'course': '数字电路与模拟电路', 'sports': '羽毛球', 'food': '紫光园奶皮子酸奶'}, created_at='2026-07-25T14:49:18.471274+00:00', updated_at='2026-07-25T14:49:18.471276+00:00', score=0.5925544611349742)
Item(namespace=['users', 'Bob', 'memories'], key='preferences', value={'course': '数字电路与模拟电路', 'sports': '跑步', 'food': '奶皮子糖葫芦'}, created_at='2026-07-25T14:49:18.338932+00:00', updated_at='2026-07-25T14:49:18.338934+00:00', score=0.46505767961460387)
